# 🚀 Prédiction RUL — CMAPSS FD001 avec LSTM
**Pipeline complet** : Upload → Preprocessing → Feature Engineering → LSTM → Évaluation


## 📦 1. Installation des librairies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print('✅ Librairies importées')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📁 2. Upload du fichier train_FD001.txt

In [ ]:
from google.colab import files

print('📂 Uploadez votre fichier train_FD001.txt')
uploaded = files.upload()

# Récupérer le nom du fichier uploadé
file_name = list(uploaded.keys())[0]
print(f'\n✅ Fichier uploadé : {file_name}')

## 🔧 3. Chargement et nommage des colonnes

In [ ]:
# Colonnes CMAPSS
columns = [
    'unit_id', 'cycle',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    'sensor_1',  'sensor_2',  'sensor_3',  'sensor_4',  'sensor_5',
    'sensor_6',  'sensor_7',  'sensor_8',  'sensor_9',  'sensor_10',
    'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
    'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20',
    'sensor_21'
]

df = pd.read_csv(file_name, sep=' ', header=None, names=columns, index_col=False)

# Supprimer les colonnes vides (le fichier a parfois des espaces en trop)
df.dropna(axis=1, how='all', inplace=True)
df.dropna(inplace=True)

print(f'✅ Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes')
print(f'   Nombre de moteurs  : {df["unit_id"].nunique()}')
print(f'   Cycles min / max   : {df["cycle"].min()} / {df["cycle"].max()}')
df.head()

## 📊 4. Calcul du RUL (Remaining Useful Life)

In [ ]:
# Max cycle par moteur = durée de vie totale
max_cycles = df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.columns = ['unit_id', 'max_cycle']

df = df.merge(max_cycles, on='unit_id')

# RUL = max_cycle - cycle actuel
df['RUL'] = df['max_cycle'] - df['cycle']

# ✂️ Clip RUL : on plafonne à 125 (recommandation NASA pour FD001)
RUL_MAX = 125
df['RUL_clipped'] = df['RUL'].clip(upper=RUL_MAX)

print(f'✅ RUL calculé')
print(f'   RUL min  : {df["RUL"].min()}')
print(f'   RUL max  : {df["RUL"].max()}')
print(f'   RUL moyen: {df["RUL"].mean():.1f}')

# Visualisation de la distribution RUL
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(df['RUL'], bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution RUL (brut)')
plt.xlabel('RUL'); plt.ylabel('Fréquence')

plt.subplot(1, 2, 2)
plt.hist(df['RUL_clipped'], bins=50, color='coral', edgecolor='white')
plt.title(f'Distribution RUL (clippé à {RUL_MAX})')
plt.xlabel('RUL'); plt.ylabel('Fréquence')
plt.tight_layout()
plt.show()

## 🔬 5. Sélection des capteurs utiles
Certains capteurs sont constants → on les supprime (variance ≈ 0)

In [ ]:
# Capteurs inutiles pour FD001 (variance nulle)
DROP_SENSORS = ['sensor_1', 'sensor_5', 'sensor_6', 'sensor_10',
                'sensor_16', 'sensor_18', 'sensor_19']

# Feature columns
sensor_cols = [c for c in columns if 'sensor' in c and c not in DROP_SENSORS]
op_cols     = ['op_setting_1', 'op_setting_2', 'op_setting_3']
feature_cols = op_cols + sensor_cols

print(f'✅ Capteurs sélectionnés : {len(sensor_cols)}')
print(f'   Features totales      : {len(feature_cols)}')
print(f'   Features : {feature_cols}')

# Vérification des variances
variances = df[sensor_cols].var()
plt.figure(figsize=(12, 3))
variances.plot(kind='bar', color='teal')
plt.title('Variance des capteurs sélectionnés')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## ⚙️ 6. Normalisation des features (MinMaxScaler)

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

df[feature_cols] = scaler_X.fit_transform(df[feature_cols])
df['RUL_scaled']  = scaler_y.fit_transform(df[['RUL_clipped']])

print('✅ Normalisation effectuée (MinMaxScaler [0, 1])')

## 🏗️ 7. Création des séquences temporelles pour le LSTM
Le LSTM prend en entrée une **fenêtre glissante** de `window_size` cycles

In [ ]:
WINDOW_SIZE = 30  # nombre de cycles dans la fenêtre

def create_sequences(data, feature_cols, target_col, window_size):
    X, y = [], []
    for unit_id in data['unit_id'].unique():
        unit_data = data[data['unit_id'] == unit_id].sort_values('cycle')
        features  = unit_data[feature_cols].values
        targets   = unit_data[target_col].values
        
        if len(unit_data) < window_size:
            # Moteur trop court : on pad avec des zéros
            pad_len  = window_size - len(unit_data)
            features = np.vstack([np.zeros((pad_len, len(feature_cols))), features])
            targets  = np.hstack([np.zeros(pad_len), targets])
        
        for i in range(len(features) - window_size + 1):
            X.append(features[i:i + window_size])
            y.append(targets[i + window_size - 1])
    
    return np.array(X), np.array(y)

X, y = create_sequences(df, feature_cols, 'RUL_scaled', WINDOW_SIZE)

print(f'✅ Séquences créées')
print(f'   X shape : {X.shape}  → (nb_séquences, window_size, nb_features)')
print(f'   y shape : {y.shape}')

## ✂️ 8. Split Train / Validation (80 / 20)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f'✅ Split effectué')
print(f'   Train      : {X_train.shape[0]} séquences')
print(f'   Validation : {X_val.shape[0]} séquences')

## 🧠 9. Construction du modèle LSTM

In [ ]:
def build_lstm_model(input_shape):
    model = Sequential([
        # 1er bloc LSTM
        LSTM(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),
        
        # 2ème bloc LSTM
        LSTM(64, return_sequences=True),
        BatchNormalization(),
        Dropout(0.3),
        
        # 3ème bloc LSTM
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        
        # Couches denses
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        
        # Sortie : valeur continue (régression)
        Dense(1, activation='sigmoid')  # sigmoid car RUL normalisé [0,1]
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

model = build_lstm_model((WINDOW_SIZE, len(feature_cols)))
model.summary()

## 🏋️ 10. Entraînement du modèle

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=7, min_lr=1e-6, verbose=1
)

# Entraînement
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print('\n✅ Entraînement terminé !')

## 📈 11. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'],     label='Train Loss',      color='steelblue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='coral')
axes[0].set_title('Loss (MSE) pendant l\'entraînement')
axes[0].set_xlabel('Epochs'); axes[0].set_ylabel('MSE')
axes[0].legend()

# MAE
axes[1].plot(history.history['mae'],     label='Train MAE',      color='steelblue')
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='coral')
axes[1].set_title('MAE pendant l\'entraînement')
axes[1].set_xlabel('Epochs'); axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📊 12. Évaluation sur le set de validation

In [ ]:
# Prédictions sur la validation
y_pred_scaled = model.predict(X_val).flatten()

# Dé-normalisation
y_pred_real = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_val_real  = scaler_y.inverse_transform(y_val.reshape(-1, 1)).flatten()

# Métriques
rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
mae  = mean_absolute_error(y_val_real, y_pred_real)

# Score NASA (pénalise les sous-estimations plus que les sur-estimations)
def nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    score = np.sum(np.where(diff < 0,
                            np.exp(-diff / 13) - 1,
                            np.exp(diff  / 10) - 1))
    return score

nasa = nasa_score(y_val_real, y_pred_real)

print('=' * 40)
print('📊 RÉSULTATS SUR VALIDATION')
print('=' * 40)
print(f'  RMSE        : {rmse:.2f} cycles')
print(f'  MAE         : {mae:.2f} cycles')
print(f'  NASA Score  : {nasa:.0f}  (plus bas = meilleur)')
print('=' * 40)

## 🎯 13. Visualisation : RUL prédit vs réel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_val_real, y_pred_real, alpha=0.3, color='steelblue', s=10)
axes[0].plot([0, RUL_MAX], [0, RUL_MAX], 'r--', linewidth=2, label='Prédiction parfaite')
axes[0].set_xlabel('RUL Réel'); axes[0].set_ylabel('RUL Prédit')
axes[0].set_title(f'RUL Prédit vs Réel\nRMSE={rmse:.1f}, MAE={mae:.1f}')
axes[0].legend()

# Distribution des erreurs
errors = y_pred_real - y_val_real
axes[1].hist(errors, bins=50, color='teal', edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Erreur (cycles)'); axes[1].set_ylabel('Fréquence')
axes[1].set_title(f'Distribution des erreurs\nMoyenne={errors.mean():.1f}, Std={errors.std():.1f}')

plt.tight_layout()
plt.show()

## 🔮 14. Prédiction sur un moteur (exemple)

In [ ]:
# Choisir un moteur aléatoire pour visualiser la prédiction cycle par cycle
unit_example = np.random.choice(df['unit_id'].unique())

unit_data = df[df['unit_id'] == unit_example].sort_values('cycle')
features  = unit_data[feature_cols].values
rul_true  = unit_data['RUL_clipped'].values

# Créer les séquences pour ce moteur
X_unit, y_unit = [], []
for i in range(len(features) - WINDOW_SIZE + 1):
    X_unit.append(features[i:i + WINDOW_SIZE])

X_unit = np.array(X_unit)
y_pred_unit = model.predict(X_unit, verbose=0).flatten()
y_pred_unit = scaler_y.inverse_transform(y_pred_unit.reshape(-1, 1)).flatten()

# Cycles correspondants
cycles = unit_data['cycle'].values[WINDOW_SIZE - 1:]
rul_true_unit = rul_true[WINDOW_SIZE - 1:]

plt.figure(figsize=(12, 5))
plt.plot(cycles, rul_true_unit, label='RUL Réel',  color='steelblue', linewidth=2)
plt.plot(cycles, y_pred_unit,   label='RUL Prédit', color='coral',     linewidth=2, linestyle='--')
plt.axhline(y=0, color='red', linestyle=':', linewidth=1, label='Panne')
plt.fill_between(cycles, rul_true_unit, y_pred_unit, alpha=0.2, color='orange')
plt.xlabel('Cycle'); plt.ylabel('RUL (cycles)')
plt.title(f'Moteur #{unit_example} — RUL Réel vs Prédit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nMoteur #{unit_example} :')
print(f'  Durée totale       : {unit_data["cycle"].max()} cycles')
print(f'  RMSE sur ce moteur : {np.sqrt(mean_squared_error(rul_true_unit, y_pred_unit)):.2f} cycles')

## 💾 15. Sauvegarde du modèle

In [ ]:
model.save('lstm_rul_fd001.h5')
print('✅ Modèle sauvegardé : lstm_rul_fd001.h5')

# Télécharger le modèle
files.download('lstm_rul_fd001.h5')

print('\n🎉 Pipeline complet terminé !')
print('=' * 40)
print(f'  RMSE       : {rmse:.2f} cycles')
print(f'  MAE        : {mae:.2f} cycles')
print(f'  NASA Score : {nasa:.0f}')
print('=' * 40)